In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv

In [2]:
import os
load_dotenv()
EXCHANGE_RATE_API = os.getenv("EXCHANGE_RATE_API")


In [3]:
# tool create
# 2 tools -> 1 for conversion ; 2 multiplication with value

@tool
def get_conversion_factor(base_currency:str)->float :
    '''This function fetches the currency conversion factor b/w base currency and target curency '''
    url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API}/latest/{base_currency}"
    response = requests.get(url)
    return response.json()
@tool
def convert(base_currency_value:int , conversion_rate:float)-> float:
    '''Given a currency conversion rate this function calculates the target currency value from a given base currency value'''

    return base_currency_value * conversion_rate


In [4]:
result = get_conversion_factor.invoke({'base_currency':'USD'})
result

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1785110401,
 'time_last_update_utc': 'Mon, 27 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1785196801,
 'time_next_update_utc': 'Tue, 28 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'conversion_rates': {'USD': 1,
  'AED': 3.6725,
  'AFN': 66.022,
  'ALL': 82.3937,
  'AMD': 365.8114,
  'ANG': 1.79,
  'AOA': 921.3527,
  'ARS': 1494.3744,
  'AUD': 1.4304,
  'AWG': 1.79,
  'AZN': 1.7009,
  'BAM': 1.7173,
  'BBD': 2.0,
  'BDT': 123.6077,
  'BGN': 1.7173,
  'BHD': 0.376,
  'BIF': 2990.8731,
  'BMD': 1.0,
  'BND': 1.2902,
  'BOB': 11.1515,
  'BRL': 5.0861,
  'BSD': 1.0,
  'BTN': 96.6315,
  'BWP': 13.7969,
  'BYN': 2.8735,
  'BZD': 2.0,
  'CAD': 1.4087,
  'CDF': 2289.5607,
  'CHF': 0.8168,
  'CLF': 0.02395,
  'CLP': 946.5279,
  'CNH': 6.7702,
  'CNY': 6.7777,
  'COP': 3207.1885,
  'CRC': 455.1568,
  'CUP': 24.0,
  'CVE':

In [5]:
convert.invoke({'base_currency_value':10,'conversion_rate':96.6138})

966.1379999999999

In [6]:
#Step 2 -> tool binding

In [7]:
llm = ChatGoogleGenerativeAI(model = 'gemini-3.6-flash')

In [8]:
llm_with_tools= llm.bind_tools([convert,get_conversion_factor])

In [9]:
# tool calling
messages=[HumanMessage('What is the conversion factor between usd and inr and based on that can u convert 10 usd to inr')]

In [10]:
ai_msg = llm_with_tools.invoke(messages)
ai_msg

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'J5pMWlD0': 'EvgGCvUGARFNMg/qmG99P8P8xUFxPOYwy8BmBR4ZHYHo3zGC8JmfTcsZ+R9BI/NZx3tuXM7PFtwvNoUpKUjD/VOlt457jBtbXV4k5szAve44+Igb8fcF4FMel7vzBugI2NNKX4ZsTYVKvfcAmn/q1X+hRDu1PL3Y1904+Yb6nNMAWwUGVJa6vTDt2dkTW38a84WA8NIFbnligszW+pgzuvVuIFWFOWBJtTxBcpMWzmDOsfjS000toUlZlCoGGPFR3wn83JYLGh3/N4WjjJt1F8jBvRaGqeEhZKJRiq5Mg3bnV6mA+AYilUhI/GZy310HQi240jjI54162i6P/3PInGota5aJbTUQsawC8CGrD2QexhxQjlEv4zeuv90FTYIlYY5tgaJweEGtmWfoJuyj5gIWsapxGKhZrH84Z5sVQ+IygBT/zWmZ5tkGmNt/TyBusSheEGFpHTyaose2pMjDDLnyHRQUoBLi/y2JygUOSGcvMZOz+dz91HDu6MJ4cPzq4pdut6CXGnHjQRhB7kl/RxNY1eYSsdzYPbwlbVzkZdGzK5g+65L1J1ssbtp1zhRiNqk/GqsA65yN8/So8k/wmAYh2DLOv4VdX8fPiqYwkCIGXxMgbCmRRSdS3I+nFELQJlQqA5s3DzhEoWphWzF0k8kv/u5CLNSXw60GiXazxmZoLAisGj4mEVMU9uqaxyv0WV5ojWT94ZXtOqPdUoGa1CUe36kDY6SFZkrqVzQ+kNxiOsGktl9sr6E84YTkfM0Oociw7mIzGRc2LUjrMcop0PJAPnRsKRsuqhP0F